In [2]:
import torch
import torch.nn as nn
from PatchEmbed import PatchEmbed
from MultiHeadAttention import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        # 1. LayerNorm before attention
        self.norm1 = nn.LayerNorm(embed_dim)
        # 2. MultiHeadAttention
        self.MHA = MultiHeadAttention(embed_dim, num_heads, dropout)
        # 3. LayerNorm before MLP
        self.norm2 = nn.LayerNorm(embed_dim)
        # 4. MLP: Linear → GELU → Dropout → Linear → Dropout
        #    (hidden size = int(embed_dim * mlp_ratio))
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # residual around attention
        x = x + self.MHA(self.norm1(x))

        # residual around MLP
        x = x + self.mlp(self.norm2(x))

        return x

In [3]:
block = TransformerBlock(embed_dim=128, num_heads=4)
x = torch.randn(2, 64, 128)   # batch, num_patches, embed_dim
out = block(x)
print(out.shape)               # must stay [2, 64, 128]

torch.Size([2, 64, 128])
